In [ ]:
import pandas as pd
import simplejson
import json

pd.set_option('display.max_columns', None)

In [69]:
#Specify periods to be removed
PeriodRemovals = "2024-25"
PeriodRemovals = PeriodRemovals.split(";")
MoreSpecificRemovals = [{'Code': 'CORP 09', 'Period': '2023-24'},{'Code': 'CORP 10', 'Period': '2023-24'}]

In [70]:
AllData = pd.read_csv('Data Files//Indicator Data - Real - Annual.csv', converters={'Changes': json.loads,'Previous_Row': json.loads}).query(f'LocalAuthority == "Stirling"')
AllData = AllData[~AllData.Period.isin(PeriodRemovals)]
for i in MoreSpecificRemovals :
    dropIndexes = AllData[(AllData['Period'] == i['Period']) & (AllData['Code'] == i['Code'])].index
    AllData.drop(dropIndexes, inplace=True)

In [71]:
LatestData = AllData.copy(deep=True)
LatestData.sort_values(by=['LocalAuthority', 'Code', 'Period'], inplace=True)
LatestData = LatestData.groupby(['LocalAuthority', 'Code']).tail(1)

In [72]:
def PreviousConvertToJson(df):
    Previous_Row = simplejson.dumps(df['Previous_Row'], ignore_nan=True)
    return Previous_Row


def FirstConvertToJson(df):
    First_Row = simplejson.dumps(df['First_Row'], ignore_nan=True)
    return First_Row


def ChangesConvertToJson(df):
    Changes = simplejson.dumps(df['Changes'], ignore_nan=True)
    return Changes

LatestData['Previous_Row'] = LatestData.apply(PreviousConvertToJson, axis=1)
LatestData['First_Row'] = LatestData.apply(FirstConvertToJson, axis=1)
LatestData['Changes'] = LatestData.apply(ChangesConvertToJson, axis=1)

In [73]:
LatestData.to_csv('test.csv',index=False, encoding='utf-8-sig')